# 52 — DeepEval Synthesizer: Auto-Generate Golden Datasets
## Generate Your Own Test Set — Synthetic Golden Datasets
⏱ ~15 min

**Manual golden dataset creation is the #1 bottleneck in LLM evaluation adoption.** Writing 100 question-answer pairs from scratch takes 4–8 hours of expert time, and the dataset goes stale every time your documents change. DeepEval's **Synthesizer** eliminates this bottleneck by auto-generating diverse, high-quality test cases from your documents.

By the end of this session you will understand *why* synthetic goldens work, *how* to generate and evolve them, and *how* to use them for regression testing that actually catches pipeline regressions.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Why Synthesizer?** — The manual golden bottleneck |
| 2 | **Generate Goldens** — `Synthesizer.generate_goldens_from_docs()` |
| 3 | **Evolution Strategies** — SIMPLE, REASONING, MULTI_HOP, COMPARATIVE |
| 4 | **Save & Load** — `EvaluationDataset.save()` / `.load()` |
| 5 | **Baseline Evaluation** — Run metrics on generated goldens |
| 6 | **Regression Testing** — Detect score drops after pipeline changes |
| 7 | **Manual vs Synthesized** — Side-by-side comparison |
| ★ | **Exercises + Answer Key** |

---

### Prerequisites
- Python 3.10+ with a `.venv` containing the project requirements
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)
- Familiarity with basic RAG concepts (see `examples/12-basic-rag-notebook/`)
- Familiarity with DeepEval metrics (see `examples/47-deepeval-rag-metrics/`)

### Key References
> Confident AI. (2024). *DeepEval: The Open-Source Evaluation Framework for LLMs.*  
> https://github.com/confident-ai/deepeval
>
> Chang, Y., Wang, X., et al. (2024). *A Survey on Evaluation of Large Language Models.*  
> https://arxiv.org/abs/2307.03109
>
> DeepEval Synthesizer docs: https://docs.confident-ai.com/docs/synthesizer-introduction

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "deepeval",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "chromadb",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key_ok = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY present: {key_ok}")
assert key_ok, (
    "Set OPENAI_API_KEY before running.\n"
    "  Local: echo 'OPENAI_API_KEY=sk-...' >> .env\n"
    "  Colab: Secrets panel -> Add secret 'OPENAI_API_KEY'"
)

---

## Part 1 — Why Synthesizer? The Manual Golden Bottleneck

---

### The Problem with Manual Evaluation Sets

A **golden dataset** is a curated set of (question, expected_answer, context) triples used to evaluate your RAG pipeline. Without one, you cannot measure whether your pipeline is getting better or worse.

The traditional approach is manual: a domain expert reads your documents, writes questions, crafts expected answers, and tags which context chunk should be retrieved. At 100 goldens — a modest set for any real product — that is:

| Task | Time per golden | Total (100 goldens) |
|------|-----------------|---------------------|
| Write question | ~2 min | ~3.3 hours |
| Write expected answer | ~3 min | ~5 hours |
| Tag context chunk | ~1 min | ~1.7 hours |
| Review for quality | ~2 min | ~3.3 hours |
| **Total** | **~8 min** | **~13 hours** |

And when documents are updated, the goldens go stale. The dataset must be regenerated.

### Synthesizer Approach

```
Your documents
     │
     ▼
Synthesizer.generate_goldens_from_docs()
     │
     ├─── [SIMPLE]      Direct factual question from one context
     ├─── [REASONING]   Requires inference across context
     ├─── [MULTI_HOP]   Links information across multiple contexts
     └─── [COMPARATIVE] Asks model to compare two or more ideas
     │
     ▼
EvaluationDataset  (goldens with input + expected_output + context)
     │
     ▼  .save()
goldens_baseline.json
     │
     │  (run pipeline changes)
     │
     ▼  .load()
Re-evaluate same goldens
     │
     ▼
Delta comparison: baseline vs degraded
     AnswerRelevancy: 0.82 → 0.61  (delta: -0.21)  ← REGRESSION
     Faithfulness:   0.79 → 0.74  (delta: -0.05)  ← marginal
```

### Key Vocabulary

| Term | Definition |
|------|------------|
| **Golden** | A test case with a known-good expected output |
| **Evolution strategy** | How the synthesizer mutates a base question to be harder |
| **EvaluationDataset** | DeepEval's container for a collection of goldens |
| **Regression test** | Re-running the same goldens before and after a pipeline change |
| **Delta** | The numeric difference between baseline and degraded scores |
| **Faithfulness** | Does the answer stick to what the retrieved context says? |
| **AnswerRelevancy** | Does the answer actually address the question asked? |

In [ ]:
# ── 1-A: The bottleneck illustrated ──────────────────────────────────────────
#
# These are the documents our workshop RAG pipeline knows about.
# Six short passages — enough to generate diverse goldens and test retrieval.

DOCUMENTS = [
    "LangGraph is a library for building stateful, multi-actor applications with LLMs. "
    "It uses a graph-based workflow where nodes are Python functions and edges control flow.",
    "LangChain provides modular components for building LLM applications: chains, agents, "
    "retrievers, and output parsers. It integrates with 50+ LLM providers.",
    "RAG (Retrieval-Augmented Generation) improves LLM accuracy by retrieving relevant "
    "documents before generation. Key metrics: Faithfulness, AnswerRelevancy, ContextualPrecision.",
    "Vector databases store numerical embeddings for semantic similarity search. "
    "Popular options: ChromaDB (local), Pinecone (cloud), Weaviate (self-hosted).",
    "Agents use LLMs as reasoning engines to decide which tools to call and when. "
    "ReAct (Reason + Act) is the dominant pattern: think step-by-step, then act.",
    "Checkpointing in LangGraph allows saving and resuming agent state. "
    "SqliteSaver persists to disk; MemorySaver keeps state in memory only.",
]

# How many manual goldens would this cost?
MIN_GOLDENS = 20  # reasonable minimum for any evaluation
minutes_per_golden = 8
hours = (MIN_GOLDENS * minutes_per_golden) / 60
print(f"Manual approach: {MIN_GOLDENS} goldens x {minutes_per_golden} min = {hours:.1f} hours")
print(f"Synthesizer approach: generate_goldens_from_docs() -> seconds")
print(f"\n{len(DOCUMENTS)} documents loaded — ready to synthesize.")

---

## Part 2 — Generate Goldens from Documents

---

### How `generate_goldens_from_docs` Works

The Synthesizer takes a list of context strings (your document chunks) and for each context:

1. Generates a base question that can be answered from the context
2. Evolves the question using the requested strategy (harder, more nuanced)
3. Generates the expected answer using the context as ground truth
4. Packages the triple as a `Golden` object

The result is an `EvaluationDataset` where every golden has:
- `input`: the generated question
- `expected_output`: the generated reference answer
- `context`: the source document chunk(s)

### API Quick Reference

```python
from deepeval.synthesizer import Synthesizer
from deepeval.dataset import EvaluationDataset

from copy import copy
from deepeval.models import GPTModel

def deepeval_gpt54_nano() -> GPTModel:
    """Enable DeepEval's typed-output path for this newer OpenAI model."""
    model = GPTModel(model="gpt-5.4-nano")
    model.model_data = copy(model.model_data)
    model.model_data.supports_structured_outputs = True
    model.calculate_cost = lambda _input_tokens, _output_tokens: 0.0
    return model

synthesizer = Synthesizer(model=deepeval_gpt54_nano(), async_mode=False)

# Generate from inline context lists
goldens = synthesizer.generate_goldens_from_docs(
    document_paths=None,          # None when passing contexts inline
    contexts=[[doc] for doc in documents],
    max_goldens_per_context=2,    # how many Q&A pairs per chunk
)

# Or load from files
goldens = synthesizer.generate_goldens_from_docs(
    document_paths=["./docs/policy.pdf", "./docs/guide.txt"],
    max_goldens_per_context=3,
)
```

In [ ]:
# ── 2-A: Generate goldens from inline contexts ────────────────────────────────
from deepeval.dataset import EvaluationDataset
from deepeval.synthesizer import Synthesizer

from copy import copy
from deepeval.models import GPTModel

def deepeval_gpt54_nano() -> GPTModel:
    """Enable DeepEval's typed-output path for this newer OpenAI model."""
    model = GPTModel(model="gpt-5.4-nano")
    model.model_data = copy(model.model_data)
    model.model_data.supports_structured_outputs = True
    model.calculate_cost = lambda _input_tokens, _output_tokens: 0.0
    return model

synthesizer = Synthesizer(model=deepeval_gpt54_nano(), async_mode=False)

# Each document becomes its own context list.
# max_goldens_per_context=2 -> 2 Q&A pairs per document -> ~12 goldens total.
goldens = synthesizer.generate_goldens_from_contexts(
    contexts=[[doc] for doc in DOCUMENTS],
    max_goldens_per_context=2,
)

print(f"Generated {len(goldens)} goldens from {len(DOCUMENTS)} documents")
print()
print("Sample goldens:")
for i, g in enumerate(goldens[:4]):
    print(f"  [{i}] Q: {g.input[:90]}")
    print(f"       A: {g.expected_output[:90]}")
    print()

---

## Part 3 — Evolution Strategies

---

### What Are Evolution Strategies?

A naive synthesizer just asks "generate a question from this text" and gets surface-level factual questions. **Evolution strategies** take a base question and deliberately make it harder or more nuanced — producing a more realistic test distribution.

Think of it like the difference between a quiz and a real exam:
- Quiz: "What does RAG stand for?" (recall)
- Exam: "Given that RAG retrieves before generating, how would you expect its faithfulness score to compare to a model that only uses its training knowledge, and why?" (reasoning)

### Evolution Strategy Comparison

| Strategy | Description | Difficulty | Best for |
|----------|-------------|------------|----------|
| `SIMPLE` | Direct factual question from one context chunk | Easy | Baseline coverage, smoke tests |
| `REASONING` | Requires inference or logic beyond stated facts | Medium | Testing comprehension depth |
| `MULTI_HOP` | Links information across multiple context chunks | Hard | Complex knowledge graphs |
| `COMPARATIVE` | Asks model to compare two or more entities | Hard | Evaluation of nuanced understanding |

### How Evolution Works (Conceptually)

```
Source context:
  "LangGraph is a library for building stateful, multi-actor applications."

SIMPLE:       "What is LangGraph used for?"
                └─ direct extraction from context

REASONING:    "Why might a stateful approach be important for multi-actor LLM apps?"
                └─ requires logical inference beyond what is literally stated

MULTI_HOP:    "What LangGraph feature helps address the checkpointing needs described
               in the checkpointing document?"
                └─ must connect info across two separate context chunks

COMPARATIVE:  "How does LangGraph's graph-based workflow compare to LangChain's
               chain-based approach for building LLM applications?"
                └─ requires synthesizing two separate context chunks
```

> **Rule of thumb:** Start with SIMPLE goldens to verify basic retrieval works, then add REASONING and MULTI_HOP goldens to expose weaknesses in comprehension and cross-chunk retrieval.

In [ ]:
# ── 3-A: Inspect goldens by evolution type ────────────────────────────────────
# The Synthesizer assigns an evolution type to each golden.
# Inspect the distribution to understand what was generated.

# Check what attributes goldens have
sample_golden = goldens[0]
print("Golden attributes:", [attr for attr in dir(sample_golden) if not attr.startswith("_")])
print()

# Print all goldens with their evolution type
print(f"{'#':<4} {'Evolution':<15} {'Question (truncated)':<60}")
print("-" * 79)
for i, g in enumerate(goldens):
    evolution = getattr(g, "evolution", getattr(g, "synthesizer_evolution", "UNKNOWN"))
    print(f"{i:<4} {str(evolution):<15} {g.input[:58]}")

print()
print("Tip: MULTI_HOP and REASONING goldens are harder — they expose deeper pipeline weaknesses.")

---

## Part 4 — Save and Load: EvaluationDataset

---

### Why Persist Goldens?

The critical insight for regression testing: **you must use exactly the same goldens before and after a pipeline change.** If you regenerate goldens, you cannot tell whether a score drop is due to the pipeline or the different questions.

```python
# Save after generating (run once)
dataset = EvaluationDataset(test_cases=test_cases)
dataset.save("goldens_baseline.json")

# Load before evaluating (every regression run)
dataset = EvaluationDataset()
dataset.load("goldens_baseline.json")
```

The JSON file is human-readable — you can inspect and manually edit goldens if any look wrong before committing them to your test suite.

### Regression Testing Pattern

```
Week 1: generate goldens -> save -> evaluate -> record baseline scores
Week 2: change retrieval (k=1, new chunking, model swap)
         -> load same goldens -> evaluate -> compare to baseline
         -> delta reveals whether the change helped or hurt
Week 3: fix regression -> load same goldens -> evaluate -> confirm recovery
```

**The goldens are your regression test suite.** Treat them like unit tests — version control the JSON, do not delete it unless documents change.

In [ ]:
# ── 4-A: Build the RAG pipeline ───────────────────────────────────────────────
# Build a simple RAG pipeline we will use for baseline evaluation and regression.

from deepeval.test_case import LLMTestCase
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0.2)
vectorstore = Chroma.from_texts(
    DOCUMENTS,
    embeddings,
    collection_name="synth_eval",
)

ANSWER_PROMPT = (
    "Answer using ONLY the context below. "
    "If the context does not contain the answer, say 'I don't know.'\n\n"
    "Context:\n{context}\n\nQuestion: {query}"
)


def run_rag(query: str, k: int = 3) -> tuple[str, list[str]]:
    """Run RAG pipeline. Returns (answer, retrieved_context_list)."""
    docs = vectorstore.similarity_search(query, k=k)
    context = [d.page_content for d in docs]
    context_text = "\n".join(f"- {c}" for c in context)
    answer = llm.invoke(
        [HumanMessage(content=ANSWER_PROMPT.format(context=context_text, query=query))]
    ).content
    return answer, context


print("RAG pipeline ready.")
print(f"Vector store: {len(DOCUMENTS)} documents indexed with text-embedding-3-small")

# Smoke test
test_answer, test_ctx = run_rag("What is LangGraph?", k=3)
print(f"\nSmoke test: 'What is LangGraph?'")
print(f"Answer: {test_answer[:100]}...")

---

## Part 5 — Baseline Evaluation

---

### From Goldens to Test Cases

A `Golden` contains the synthesized question and expected answer. To evaluate the RAG pipeline, we need to:

1. Run each golden's `input` through the RAG pipeline
2. Collect the `actual_output` (what the pipeline actually said)
3. Collect the `retrieval_context` (which chunks were retrieved)
4. Package these into `LLMTestCase` objects
5. Run metrics against the test cases

```
Golden:
  input           = "What is LangGraph used for?"
  expected_output = "LangGraph is used for building stateful, multi-actor apps..."
  context         = ["LangGraph is a library..."]   <- source chunk

LLMTestCase (after running pipeline):
  input             = "What is LangGraph used for?"   (same)
  expected_output   = "LangGraph is used for..."      (same)
  actual_output     = "LangGraph allows you to..."    (what RAG returned)
  retrieval_context = ["LangGraph is a library..."]   (chunks RAG retrieved)
```

In [ ]:
# ── 5-A: Build baseline test cases from goldens ───────────────────────────────
from deepeval import evaluate
from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric

metrics = [
    AnswerRelevancyMetric(threshold=0.7, model=deepeval_gpt54_nano()),
    FaithfulnessMetric(threshold=0.7, model=deepeval_gpt54_nano()),
]

print("Building baseline test cases (k=3)...")
test_cases_k3 = []
for g in goldens:
    answer, context = run_rag(g.input, k=3)
    test_cases_k3.append(
        LLMTestCase(
            input=g.input,
            actual_output=answer,
            expected_output=g.expected_output,
            retrieval_context=context,
        )
    )

print(f"Built {len(test_cases_k3)} test cases")
print()

# Run baseline evaluation
print("Running baseline evaluation...")
results_k3 = evaluate(test_cases_k3, metrics, run_async=False)


def avg_score(results, metric_idx: int) -> float:
    """Average a metric score across all test results."""
    scores = [tc.metrics_data[metric_idx].score for tc in results.test_results]
    return sum(scores) / len(scores)


baseline_ar = avg_score(results_k3, 0)
baseline_f = avg_score(results_k3, 1)

print(f"\nBaseline scores (k=3, n={len(test_cases_k3)} goldens):")
print(f"  AnswerRelevancy : {baseline_ar:.3f}")
print(f"  Faithfulness    : {baseline_f:.3f}")

In [ ]:
# ── 5-B: Save baseline dataset ────────────────────────────────────────────────
# Persist test cases to JSON — we will load these again for regression testing.
# IMPORTANT: use the same file every time to ensure score comparisons are valid.

import json
from pathlib import Path

dataset = EvaluationDataset(test_cases=test_cases_k3)
dataset.save("goldens_baseline.json")
print("Saved baseline dataset to goldens_baseline.json")

# Inspect the saved file structure
saved_raw = Path("goldens_baseline.json").read_text()
saved = json.loads(saved_raw)

# Handle both list and dict formats depending on DeepEval version
n_records = len(saved) if isinstance(saved, list) else len(saved.get("test_cases", []))
print(f"\nFile contents:")
print(f"  Total records: {n_records}")
print(f"  File size: {Path('goldens_baseline.json').stat().st_size / 1024:.1f} KB")
print()
print("Tip: commit goldens_baseline.json to version control.")
print("It is your regression test suite — treat it like unit tests.")

---

## Part 6 — Regression Testing: Detect Score Drops

---

### The Regression Testing Pattern

A regression test answers one question: **did this pipeline change make things worse?**

The procedure:
1. Record baseline scores on the saved goldens (done in Part 5)
2. Make a pipeline change (swap model, reduce k, change chunking)
3. Load the exact same goldens from JSON
4. Run the changed pipeline against those goldens
5. Compare scores — negative delta = regression, positive delta = improvement

### What Counts as a Regression?

| Delta | Classification | Action |
|-------|---------------|--------|
| < -0.05 | Regression | Block deployment, investigate |
| -0.05 to 0.0 | Marginal drop | Monitor, may be acceptable |
| 0.0 to +0.05 | Neutral/slight improvement | Deploy with monitoring |
| > +0.05 | Improvement | Deploy, update baseline |

> **Common regression causes:** reducing retrieval k, switching to a weaker model, changing chunking that splits key facts across chunk boundaries, adding more documents that dilute retrieval precision.

In [ ]:
# ── 6-A: Simulate a pipeline regression (k=3 -> k=1) ─────────────────────────
# We degrade the pipeline by cutting k from 3 to 1.
# k=1 means the model only sees one retrieved chunk — much less context.

# Load the saved goldens — same file, different pipeline config
loaded_dataset = EvaluationDataset()
loaded_dataset.load("goldens_baseline.json")
print(f"Loaded {len(loaded_dataset.test_cases)} test cases from goldens_baseline.json")

# Re-run with degraded retrieval (k=1)
print("\nRunning degraded pipeline (k=1)...")
test_cases_k1 = []
for tc in loaded_dataset.test_cases:
    answer, context = run_rag(tc.input, k=1)  # only 1 chunk retrieved
    test_cases_k1.append(
        LLMTestCase(
            input=tc.input,
            actual_output=answer,
            expected_output=tc.expected_output,
            retrieval_context=context,
        )
    )

results_k1 = evaluate(test_cases_k1, metrics, run_async=False)
print("Degraded evaluation complete.")

In [ ]:
# ── 6-B: Delta comparison — baseline vs degraded ──────────────────────────────


def format_delta(delta: float) -> str:
    """Format delta with classification label."""
    if delta < -0.05:
        return f"{delta:+.3f}  REGRESSION"
    elif delta < 0:
        return f"{delta:+.3f}  marginal drop"
    elif delta < 0.05:
        return f"{delta:+.3f}  neutral"
    else:
        return f"{delta:+.3f}  IMPROVEMENT"


score_k3_ar = avg_score(results_k3, 0)
score_k1_ar = avg_score(results_k1, 0)
score_k3_f = avg_score(results_k3, 1)
score_k1_f = avg_score(results_k1, 1)

print("Regression Delta Report")
print("=" * 60)
print(f"Change: k=3 -> k=1  (retrieval depth cut by 2/3)")
print(f"Goldens: {len(test_cases_k3)} test cases (identical across both runs)")
print()
print(f"{'Metric':<20} {'Baseline':>10} {'Degraded':>10} {'Delta':>25}")
print("-" * 65)
print(
    f"{'AnswerRelevancy':<20} {score_k3_ar:>10.3f} {score_k1_ar:>10.3f} "
    f"  {format_delta(score_k1_ar - score_k3_ar)}"
)
print(
    f"{'Faithfulness':<20} {score_k3_f:>10.3f} {score_k1_f:>10.3f} "
    f"  {format_delta(score_k1_f - score_k3_f)}"
)
print()
print("Interpretation:")
print("  Negative delta = regression — the change made the pipeline worse.")
print("  Fix: increase k back to 3 (or higher), re-run, confirm delta returns to ~0.")

---

## Part 7 — Manual vs Synthesized Goldens

---

### Why Synthesized Goldens Expose More Weaknesses

When humans write evaluation questions, they naturally gravitate toward questions they *know* the system can answer — the easy cases. Synthesized goldens, especially with REASONING and MULTI_HOP strategies, generate questions that are harder and more diverse.

### Side-by-Side Comparison

| Property | Hand-Written | Synthesized |
|----------|-------------|-------------|
| Creation time (100 goldens) | 8-13 hours | ~30 seconds |
| Question diversity | Low (human bias) | High (4 strategies) |
| Difficulty range | Usually easy-medium | Easy to very hard |
| Alignment with docs | Perfect (expert-curated) | High (LLM-generated) |
| Update cost when docs change | Full manual rework | Re-run Synthesizer |
| Question naturalness | High (human phrasing) | Medium-high |
| Coverage of edge cases | Low | High |

**Recommendation:** Use synthesized goldens as your primary regression test suite. Manually review a sample (10-20%) and edit any that seem wrong. This takes 30 minutes instead of 13 hours.

In [ ]:
# ── 7-A: Hand-written vs synthesized goldens — side by side ──────────────────

HAND_WRITTEN_GOLDENS = [
    {
        "input": "What is LangGraph used for?",
        "expected_output": "LangGraph is used for building stateful, multi-actor LLM applications "
        "using graph-based workflows where nodes are Python functions.",
        "context": [DOCUMENTS[0]],
    },
    {
        "input": "Name two vector databases mentioned in the documents.",
        "expected_output": "ChromaDB and Pinecone are mentioned as vector database options.",
        "context": [DOCUMENTS[3]],
    },
    {
        "input": "What does RAG stand for?",
        "expected_output": "RAG stands for Retrieval-Augmented Generation.",
        "context": [DOCUMENTS[2]],
    },
]

print("HAND-WRITTEN GOLDENS (typical human-authored style):")
print("-" * 60)
for i, g in enumerate(HAND_WRITTEN_GOLDENS):
    print(f"  [{i}] Q: {g['input']}")
    print(f"       A: {g['expected_output'][:80]}")
    print()

print("SYNTHESIZED GOLDENS (same documents, Synthesizer output):")
print("-" * 60)
for i, g in enumerate(goldens[:4]):
    evolution = getattr(g, "evolution", "UNKNOWN")
    print(f"  [{i}] [{evolution}] Q: {g.input[:80]}")
    print(f"       A: {g.expected_output[:80]}")
    print()

print("Observation: synthesized goldens include harder question types")
print("that hand-written sets typically miss — broader coverage.")

In [ ]:
# ── 7-B: Evaluate both sets and compare ───────────────────────────────────────
# Run both hand-written and synthesized through the same pipeline.
# Compare average AnswerRelevancy to see which set is more challenging.

hw_cases = []
for g in HAND_WRITTEN_GOLDENS:
    answer, context = run_rag(g["input"], k=3)
    hw_cases.append(
        LLMTestCase(
            input=g["input"],
            actual_output=answer,
            expected_output=g["expected_output"],
            retrieval_context=context,
        )
    )

synth_cases = []
for g in goldens[: len(HAND_WRITTEN_GOLDENS)]:
    answer, context = run_rag(g.input, k=3)
    synth_cases.append(
        LLMTestCase(
            input=g.input,
            actual_output=answer,
            expected_output=g.expected_output,
            retrieval_context=context,
        )
    )

hw_results = evaluate(hw_cases, metrics, run_async=False)
synth_results = evaluate(synth_cases, metrics, run_async=False)

hw_ar = avg_score(hw_results, 0)
synth_ar = avg_score(synth_results, 0)
hw_f = avg_score(hw_results, 1)
synth_f = avg_score(synth_results, 1)

print("Hand-written vs Synthesized (same RAG pipeline, k=3)")
print("=" * 55)
print(f"{'Metric':<20} {'Hand-written':>15} {'Synthesized':>15}")
print("-" * 50)
print(f"{'AnswerRelevancy':<20} {hw_ar:>15.3f} {synth_ar:>15.3f}")
print(f"{'Faithfulness':<20} {hw_f:>15.3f} {synth_f:>15.3f}")
print()
print("Key insight: if synthesized scores are lower, the harder questions")
print("expose real weaknesses that hand-written easy questions would miss.")

---

## Exercises

---

### Exercise 1 — Generate Goldens from Your Own Documents

Replace the `MY_DOCUMENTS` list below with at least 4 documents from a domain you know (product docs, research papers, policy texts, etc.). Generate goldens and inspect them — are the questions useful? Do the expected answers look correct?

**Goal:** understand how document quality affects golden quality.

### Exercise 2 — Degrade and Measure

Using the goldens from Exercise 1 (or the workshop goldens):
1. Establish a baseline with `k=4`
2. Degrade to `k=1`
3. Degrade further: swap the LLM to a weaker model or add a hallucination-inducing prompt
4. Print the delta for each degradation

**Goal:** see how sensitive different metrics are to different types of degradation.

### Exercise 3 — Add a Document and Regenerate

Add a new document to `DOCUMENTS`, regenerate goldens, and compare:
- Did goldens appear for the new document?
- Did the baseline scores change?
- What happens to your old `goldens_baseline.json` when documents change?

**Goal:** understand the maintenance cycle for synthetic golden datasets.

### Exercise 4 — Strategy Comparison

Generate goldens twice — once with evolution forced to SIMPLE only, once with MULTI_HOP only. Evaluate both sets on the same RAG pipeline.

**Question:** Which strategy exposes more pipeline weaknesses? Why?

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
# Replace the documents below with your own content.

MY_DOCUMENTS = [
    "TODO: Replace with your own document chunk 1.",
    "TODO: Replace with your own document chunk 2.",
    "TODO: Replace with your own document chunk 3.",
    "TODO: Replace with your own document chunk 4.",
]

my_synthesizer = Synthesizer(model=deepeval_gpt54_nano(), async_mode=False)

# TODO: generate goldens from MY_DOCUMENTS
# my_goldens = my_synthesizer.generate_goldens_from_contexts(
#     contexts=[[doc] for doc in MY_DOCUMENTS],
#     max_goldens_per_context=2,
# )

# TODO: inspect and print my_goldens
# for g in my_goldens:
#     print(f"Q: {g.input}")
#     print(f"A: {g.expected_output}")
#     print()

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# Establish baseline at k=4, then degrade to k=1, print delta.

# TODO: run run_rag(g.input, k=4) on each golden to build test_cases_k4
# TODO: run evaluate(test_cases_k4, metrics)
# TODO: run run_rag(g.input, k=1) on each golden to build test_cases_k1
# TODO: run evaluate(test_cases_k1, metrics)
# TODO: print delta table using format_delta() from Part 6

# Hint: reuse the delta comparison pattern from cell-17:
# delta_ar = avg_score(results_k1, 0) - avg_score(results_k4, 0)
# print(f"AnswerRelevancy delta: {format_delta(delta_ar)}")
print("TODO: implement Exercise 2")

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
# Add a new document and regenerate goldens. Compare old vs new dataset.

EXPANDED_DOCUMENTS = DOCUMENTS + [
    # TODO: add your new document here
    "Semantic chunking splits text at meaningful boundaries rather than fixed character counts. "
    "It uses embedding similarity to detect topic shifts before splitting.",
]

# TODO: rebuild vectorstore with EXPANDED_DOCUMENTS
# TODO: generate new goldens from EXPANDED_DOCUMENTS
# TODO: check whether goldens appeared for the new document
# TODO: evaluate on baseline goldens — did adding docs change baseline scores?
print("TODO: implement Exercise 3")

In [ ]:
# ── Exercise 4 Starter ────────────────────────────────────────────────────────
# Force specific evolution strategies and compare difficulty.

# The Synthesizer accepts evolution types as a parameter.
# Check the DeepEval docs for the exact API:
# https://docs.confident-ai.com/docs/synthesizer-generate-from-docs

# TODO: generate goldens with evolution_types=["SIMPLE"]
# TODO: generate goldens with evolution_types=["MULTI_HOP"]
# TODO: evaluate both on the same pipeline
# TODO: compare average AnswerRelevancy between SIMPLE and MULTI_HOP
# TODO: answer: which type exposed more weaknesses?
print("TODO: implement Exercise 4")

---

## What's Next?

You now know how to auto-generate golden datasets and use them for regression testing. Here is where to go deeper:

### Measure with better metrics
- **Example 47 — DeepEval RAG Metrics** (`../47-deepeval-rag-metrics/`): deep dive into Faithfulness, AnswerRelevancy, and ContextualPrecision — the metrics we used in this notebook.
- **Example 49 — GEval Custom Metrics** (`../49-deepeval-geval-custom/`): define your own scoring criteria for non-standard goldens (tone, safety, format compliance).

### Compare evaluation frameworks
- **Example 16 — RAG Eval with RAGAS** (`../16-rag-eval-notebook/`): RAGAS as an alternative to DeepEval — same RAG pipeline, different metric philosophy.

### Build on the pipeline
- **Continuous evaluation**: run `evaluate()` in CI with `assert_test=True` to block deploys when scores drop below threshold.
- **Confident AI dashboard**: `deepeval login` to push results to the hosted UI — track score trends over time across pipeline versions.
- **Golden review workflow**: after generating, export to CSV, have a domain expert annotate a sample, merge corrections back before committing.

### Further reading
- DeepEval Synthesizer: https://docs.confident-ai.com/docs/synthesizer-introduction
- DeepEval CI integration: https://docs.confident-ai.com/docs/integrations-pytest
- Chang et al. (2024). *A Survey on Evaluation of LLMs.* https://arxiv.org/abs/2307.03109

---

*Workshop complete. The logical next step is example 47 — deep-dive the metrics you just applied to these goldens.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Generate Goldens from Your Own Documents

**What to look for in the output:**
- Questions should be answerable from the document text — if they reference things not in the document, the synthesizer is hallucinating context
- Expected answers should be extractable from the context — not invented
- If quality is poor, try longer, more informative document chunks (250+ words per chunk works better than 1-2 sentences)

**Common issue:** Very short documents produce shallow or repetitive questions. Add more detail to each document or combine related short passages into one chunk before passing to the synthesizer.

In [ ]:
# Sample solution for Exercise 1
SAMPLE_DOCS = [
    "PostgreSQL is an open-source relational database management system. "
    "It supports advanced features including JSON storage, full-text search, "
    "and complex SQL queries. PostgreSQL uses MVCC (Multi-Version Concurrency Control) "
    "to handle concurrent transactions without locking.",
    "Indexes in PostgreSQL improve query performance by allowing the database to find "
    "rows without scanning the entire table. B-tree indexes are the default and work "
    "for equality and range queries. GIN indexes work for full-text search and JSONB.",
    "VACUUM in PostgreSQL reclaims storage occupied by dead tuples. "
    "AUTOVACUUM runs automatically in the background. "
    "Running ANALYZE after VACUUM updates table statistics used by the query planner.",
    "Connection pooling reduces the overhead of creating new database connections. "
    "PgBouncer is the most common PostgreSQL connection pooler. "
    "It supports session, transaction, and statement pooling modes.",
]

sample_synthesizer = Synthesizer(model=deepeval_gpt54_nano(), async_mode=False)
sample_goldens = sample_synthesizer.generate_goldens_from_contexts(
    contexts=[[doc] for doc in SAMPLE_DOCS],
    max_goldens_per_context=2,
)

print(f"Generated {len(sample_goldens)} goldens from {len(SAMPLE_DOCS)} PostgreSQL docs")
for i, g in enumerate(sample_goldens):
    print(f"  [{i}] Q: {g.input}")
    print(f"       A: {g.expected_output[:80]}")
    print()

### Exercise 2 — Degrade and Measure

**Expected outcome:**
- k=4 -> k=1 should show a meaningful drop in AnswerRelevancy (more context = more relevant answers)
- Faithfulness may drop less dramatically — with fewer chunks, the model has less to be unfaithful *about*
- Swapping to a weaker LLM typically drops both metrics simultaneously

**Why AnswerRelevancy is usually more k-sensitive than Faithfulness:**  
With k=1, the model sometimes does not have enough context to answer the question fully — so answers are less relevant. But the model is still *faithful* to the single chunk it did retrieve. Faithfulness measures adherence to retrieved context, not whether the context was sufficient.

In [ ]:
# Sample solution for Exercise 2 — multi-level degradation

degradation_results = {}

for k_val in [4, 3, 2, 1]:
    cases = []
    for g in goldens:
        answer, context = run_rag(g.input, k=k_val)
        cases.append(
            LLMTestCase(
                input=g.input,
                actual_output=answer,
                expected_output=g.expected_output,
                retrieval_context=context,
            )
        )
    results = evaluate(cases, metrics, run_async=False)
    degradation_results[k_val] = {
        "ar": avg_score(results, 0),
        "faithfulness": avg_score(results, 1),
    }

baseline_k = max(degradation_results.keys())
print(f"Multi-level degradation (baseline = k={baseline_k})")
print(f"{'k':<5} {'AnswerRelevancy':>16} {'AR Delta':>12} {'Faithfulness':>14} {'F Delta':>10}")
print("-" * 60)
for k_val in sorted(degradation_results.keys(), reverse=True):
    ar = degradation_results[k_val]["ar"]
    f = degradation_results[k_val]["faithfulness"]
    ar_delta = ar - degradation_results[baseline_k]["ar"]
    f_delta = f - degradation_results[baseline_k]["faithfulness"]
    ar_d_str = "baseline" if k_val == baseline_k else f"{ar_delta:+.3f}"
    f_d_str = "baseline" if k_val == baseline_k else f"{f_delta:+.3f}"
    print(f"{k_val:<5} {ar:>16.3f} {ar_d_str:>12} {f:>14.3f} {f_d_str:>10}")

### Exercise 3 — Add a Document and Regenerate

**Expected findings:**
- New goldens should appear that reference the new document's content
- Baseline scores on old goldens should be largely unchanged (new doc does not affect old questions)
- But if the new document dilutes retrieval (the new chunks displace old ones), scores on old goldens may drop slightly

**Key takeaway:** when documents change significantly, regenerate the full golden set and establish a new baseline. Keep the old `goldens_baseline.json` in version control as a historical record.

In [ ]:
# Sample solution for Exercise 3
EXPANDED_DOCS = DOCUMENTS + [
    "Semantic chunking splits text at meaningful boundaries rather than fixed character counts. "
    "It uses embedding similarity to detect topic shifts before splitting. "
    "This produces more coherent chunks than fixed-size splitting for long, varied documents.",
]

# Rebuild vectorstore with expanded docs
expanded_store = Chroma.from_texts(
    EXPANDED_DOCS,
    embeddings,
    collection_name="synth_eval_expanded",
)

# Generate new goldens
expanded_synthesizer = Synthesizer(model=deepeval_gpt54_nano(), async_mode=False)
expanded_goldens = expanded_synthesizer.generate_goldens_from_contexts(
    contexts=[[doc] for doc in EXPANDED_DOCS],
    max_goldens_per_context=2,
)

print(f"Original: {len(goldens)} goldens from {len(DOCUMENTS)} documents")
print(f"Expanded: {len(expanded_goldens)} goldens from {len(EXPANDED_DOCS)} documents")
print()

# Check that new doc got coverage
new_doc_covered = any(
    "semantic" in g.input.lower() or "chunking" in g.input.lower()
    for g in expanded_goldens
)
print(f"New document covered in goldens: {new_doc_covered}")
print()
print("Sample golden about new document:")
for g in expanded_goldens:
    if "semantic" in g.input.lower() or "chunking" in g.input.lower():
        print(f"  Q: {g.input}")
        print(f"  A: {g.expected_output[:100]}")
        break

### Exercise 4 — Strategy Comparison

**Expected findings:**
- SIMPLE goldens should score higher — the pipeline handles direct factual questions well
- MULTI_HOP goldens should score lower — they require synthesizing information across multiple retrieved chunks, which is harder
- The difference between SIMPLE and MULTI_HOP scores tells you how much your pipeline struggles with complex reasoning

**Practical implication:** if your MULTI_HOP scores are much lower than SIMPLE scores, your chunking strategy may be splitting related information across too many chunks. Try larger chunks or a semantic chunker that keeps related sentences together.

**Note:** The exact API parameter for forcing evolution type may vary by DeepEval version — check `deepeval.__version__` and the official docs for the current syntax.

In [ ]:
# Sample solution structure for Exercise 4
# (exact evolution_types API depends on DeepEval version — check docs)

import deepeval

print(f"DeepEval version: {deepeval.__version__}")
print("Check the API for your version:")
print("  https://docs.confident-ai.com/docs/synthesizer-generate-from-docs")
print()
print("Conceptual result for SIMPLE vs MULTI_HOP comparison:")
print()
print("  SIMPLE goldens (direct factual questions):")
print("    AnswerRelevancy: typically 0.75-0.90")
print("    Faithfulness:   typically 0.80-0.95")
print()
print("  MULTI_HOP goldens (cross-chunk reasoning):")
print("    AnswerRelevancy: typically 0.55-0.75  <- lower, harder questions")
print("    Faithfulness:   typically 0.65-0.85  <- lower, more room for drift")
print()
print("  The gap reveals how much your pipeline struggles with complex reasoning.")
print("  Large gap (>0.15) -> consider larger chunks or a re-ranker.")